# Regresija: od premice do krivulje — A: prikaz (spletna trgovina)
**Akademija UI za poslovne aplikacije — Modul 2, Nadzorovano učenje v praksi**

Ta zvezek poganja predavatelj. Vajo naredite v zvezku **`B_regresija`**.

Vprašanje: **koliko dni bo trajala dostava naročila?**

### Ponovitev
- **Regresija** pomeni, da napovedujemo število. Tu: koliko dni bo trajala dostava naročila.
- Napoved damo **ob oddaji naročila**. Značilke so samo podatki, ki so takrat znani. Podatki, ki nastanejo šele po dostavi, ne smejo med značilke.
- **Časovna delitev:** model učimo na preteklosti in ga preverimo na prihodnosti.
- **Izhodišče:** vedno napovemo isto število (mediano učnih podatkov).
- **MAE** je povprečna napaka v dneh. **RMSE** velike napake šteje bolj.
- **Linearna regresija:** $\hat{y} = b_0 + \beta_1 x$ — presek $b_0$ in naklon $\beta_1$. Učenje je iskanje premice z najmanjšo napako. Značilk je lahko več.
- **Polinomska regresija:** dodamo še stolpce x², x³ … Krivulja se lahko ukrivi. Višja stopnja ni nujno boljša na novih podatkih.

### Nastavitev okolja
Zaženite spodnji dve celici (**Shift + Enter**). Ničesar ne spreminjajte. Koda prve celice je skrita.
Druga celica pokliče skupno pripravo podatkov iz `utils.modeliranje`. Veljavne pripravljene tabele ponovno uporabi.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


In [ ]:
from utils.modeliranje import poskrbi_za_modeliranje, preberi_shemo, preberi_tabelo
PODATKI = poskrbi_za_modeliranje("olist")
DATA = PODATKI.parents[2]
shema = preberi_shemo(PODATKI / "shema.json")


In [ ]:
import json
import numpy as np
import pandas as pd
from plotnine import *
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler

from utils.orodja import nastavi_slog, MODRA, ORANZNA, ZELENA, SIVA, SEME
nastavi_slog()

---
## Regresija: problem in metrike
### A1. Problem in podatki
**Cilj:** `dni_dostave` — čas od oddaje naročila do dostave, v dnevih. **Ena vrstica = eno dostavljeno naročilo.**

**Trenutek napovedi je oddaja naročila.** Za to vajo predpostavimo, da so košarica, izdelki, lokaciji kupca in prodajalca
ter obljubljeni rok takrat že znani; izvleček podatkov njihove zgodovinske razpoložljivosti ne potrjuje.
`obljubljeni_dnevi` je rok dostave, ki ga trgovina pokaže kupcu ob nakupu.
Ocene kupca in datuma dostave **ne** uporabimo kot značilki: nastaneta šele po dostavi.

> **Povezava s 1. modulom:** to je isto pravilo kot pri uhajanju podatkov, ko smo mediano računali samo na učnih podatkih.
> Model ne sme videti ničesar, česar v trenutku napovedi še ne bi vedeli.

Tabele je shranila skripta `scripts/pripravi_olist.py` v mapo `data/pripravljeno/delitve/olist/`. Tu jih samo preberemo.

<details>
<summary><b>Kako so pripravljeni podatki? (klikni)</b></summary>

Podatke je pripravila skripta `scripts/pripravi_olist.py`. Naredi isto kot zvezka predavatelja
`A2_Priprava_Olist` (mapa `03_Uvod_v_strojno_ucenje/notebooks/01_priprava_podatkov`) in
`A2_Delitev_in_manjkajoce_Olist` (mapa `03_Uvod_v_strojno_ucenje/notebooks/02_preverjanje_modela`).
Tam so vse odločitve z razlogi. Na kratko:

1. Ena vrstica = eno dostavljeno naročilo. Cilj je `dni_dostave`: čas od oddaje naročila do dostave, v dnevih z decimalkami.
2. Napoved damo ob oddaji naročila. Značilke so samo podatki, ki so takrat znani: košarica, izdelki, lokacija, čas naročila in obljubljeni rok. Ocene kupcev in plačila niso značilke.
3. `obljubljeni_dnevi` je rok dostave, ki ga trgovina pokaže kupcu ob nakupu, v dnevih od oddaje naročila.
4. Časovna delitev: učna = kupljeno in dostavljeno pred 1. 3. 2018; validacijska = kupljeno od 1. 3. do 31. 5. 2018 in dostavljeno pred 1. 6. 2018; testna = kupljeno od 1. 6. do 31. 8. 2018.
5. Naročil, ki ob meji še niso bila dostavljena, ni v nobeni množici: njihovega cilja takrat še ne bi poznali. To so predvsem počasne dostave. Validacijska množica ima zato krajše dostave kot vsa naročila iz istega obdobja; testna množica te omejitve nima. Rezultata na validacijski in na testni množici zato nista neposredno primerljiva.
6. Manjkajoče vrednosti so dopolnjene z mediano učne množice.
7. Besedilni stolpci (zvezna država, kategorija) imajo tudi stolpce 0/1, na primer `zvezna_drzava_SP`. Seznam kategorij je iz učne množice.
8. Dolgih dostav ne odstranimo. Zato merimo MAE in RMSE.
9. Značilk vnaprej ne skaliramo. Kjer je to potrebno (potence), to naredimo v zvezku, samo iz učne množice.
10. Testna množica ostane zaprta do tekmovanja na koncu regresijskega dela.

</details>

In [ ]:
ucni = preberi_tabelo(PODATKI / "train.csv", shema)
validacijski = preberi_tabelo(PODATKI / "val.csv", shema)
shema = preberi_shemo(PODATKI / "shema.json")   # opis stolpcev

print("učna:", ucni.shape, "  validacijska:", validacijski.shape)
ucni[["order_id", "datum_nakupa", "datum_znanega_izida", "obljubljeni_dnevi", "vrednost", "postnina", "zvezna_drzava", "dni_dostave"]].head(3)

**Časovna delitev.** Naročila niso naključen kup: model bomo uporabljali na *prihodnjih* naročilih.
Zato ga učimo na naročilih pred 1. 3. 2018 in preverimo na naročilih od 1. 3. do 31. 5. 2018.
Testna množica so naročila od 1. 6. do 31. 8. 2018. Ostane zaprta do tekmovanja na koncu regresijskega dela.

Pozor na podrobnost: na dan 1. 3. 2018 nekatera februarska naročila še niso bila dostavljena.
Njihovega cilja takrat še ne bi poznali, zato jih v učni množici **ni**. Enako velja za validacijsko množico in 1. 6. 2018.

In [ ]:
stolpci = ["order_id", "datum_nakupa", "datum_znanega_izida"]
oba = pd.concat([ucni[stolpci].assign(mnozica="učna"), validacijski[stolpci].assign(mnozica="validacijska")])
pregled = oba.groupby("mnozica").agg(narocil=("order_id", "size"), prvi_nakup=("datum_nakupa", "min"),
                                     zadnji_nakup=("datum_nakupa", "max"), zadnja_dostava=("datum_znanega_izida", "max"))
print("Zapisano v shemi: testna množica (zaprta) ima", shema["vrstice_po_zbirkah"]["test"], "naročil;",
      "izpuščenih, ker izid ob meji še ni bil znan:", shema["izlocenih_vrstic"])
pregled

In [ ]:
po_mesecih = oba.groupby([oba["datum_nakupa"].dt.to_period("M").dt.to_timestamp(), "mnozica"]).size().reset_index(name="narocil")

(ggplot(po_mesecih, aes(x="datum_nakupa", y="narocil", fill="mnozica"))
 + geom_col()
 + annotate("rect", xmin=pd.Timestamp("2018-05-17"), xmax=pd.Timestamp("2018-08-17"), ymin=0, ymax=7000, fill=SIVA, alpha=0.3)
 + annotate("text", x=pd.Timestamp("2018-07-02"), y=3500, label="testna\n(zaprta)", size=9)
 + scale_fill_manual(values=[MODRA, ORANZNA])
 + labs(title="Časovna delitev naročil", x="mesec nakupa", y="število naročil", fill="")
 + theme(figure_size=(9, 3.2)))

V učni množici je zadnja dostava pred 1. 3. 2018, v validacijski pred 1. 6. 2018.
Stolpca za februar in maj 2018 sta nižja: manjkajo naročila, ki do meje še niso bila dostavljena.

Za začetek uporabimo **eno samo značilko**: `obljubljeni_dnevi`.

In [ ]:
X_ucni, y_ucni = ucni[["obljubljeni_dnevi"]], ucni["dni_dostave"]
X_val, y_val = validacijski[["obljubljeni_dnevi"]], validacijski["dni_dostave"]

print(f"Dni dostave, učna množica:         povprečje {y_ucni.mean():.1f}  mediana {y_ucni.median():.1f}")
print(f"Dni dostave, validacijska množica: povprečje {y_val.mean():.1f}  mediana {y_val.median():.1f}")

V validacijski množici so dostave krajše kot v učni. Preden rečemo, da so bile dostave spomladi 2018 hitrejše,
preverimo, katera naročila v validacijski množici manjkajo.

In [ ]:
# Naročila od marca do maja 2018, ki 1. 6. 2018 še niso bila dostavljena: v validacijski množici jih ni.
delitev = pd.read_csv(PODATKI / "delitev.csv", usecols=["order_id", "razlog"], dtype={"order_id": "string"})
manjkajo = delitev.loc[delitev["razlog"] == "validacijski_izid_ob_meji_se_ni_znan", ["order_id"]]
cista = preberi_tabelo(DATA / "pripravljeno" / "olist_modeliranje.csv",
    preberi_shemo(DATA / "pripravljeno" / "olist_shema.json"), stolpci=["order_id", "dni_dostave"])
manjkajo = manjkajo.merge(cista, on="order_id")                # samo izpuščena naročila; testnih ne odpremo
vsa = pd.concat([y_val, manjkajo["dni_dostave"]])              # vsa naročila od marca do maja 2018

pd.DataFrame({"naročil": [len(y_ucni), len(y_val), len(manjkajo), len(vsa)],
              "povprečje dni dostave": [y_ucni.mean(), y_val.mean(), manjkajo["dni_dostave"].mean(), vsa.mean()]},
             index=["učna množica", "validacijska množica", "izpuščena (dostavljena po 1. 6. 2018)", "vsa naročila marec–maj 2018"]).round(1)

Izpuščena naročila so počasne dostave. Skupaj z njimi je povprečje za obdobje marec–maj podobno učnemu.
Krajše dostave v validacijski množici so torej predvsem posledica pravila delitve, ne hitrejše dostave.

> **Za tekmovanje:** testna množica tega pravila nima; v njej so vsa naročila svojega obdobja.
> Rezultat na testni množici zato ne bo neposredno primerljiv z validacijskim.
> Slabši rezultat na testni množici ni nujno pretirano prilagajanje.

### A2. Izhodišče, MAE in RMSE
Kako izmerimo, za koliko se model zmoti? $y_i$ je resnični čas dostave naročila, $\hat{y}_i$ je napoved, $n$ je število naročil.

- **MAE** (*mean absolute error*): $\text{MAE} = \frac{1}{n}\sum_i |y_i - \hat{y}_i|$ — v povprečju se zmotimo za toliko dni.
- **RMSE** (*root mean squared error*): $\text{RMSE} = \sqrt{\frac{1}{n}\sum_i (y_i - \hat{y}_i)^2}$ — napake najprej kvadriramo, zato velike napake štejejo veliko bolj.

Majhen **umetni** primer: pet naročil, pri štirih se zmotimo za 1 dan, pri enem za 16 dni.

In [ ]:
umetne_napake = np.array([1, 1, 1, 1, 16])
print("MAE: ", np.abs(umetne_napake).mean())
print("RMSE:", round(np.sqrt((umetne_napake ** 2).mean()), 2))

Ena velika napaka RMSE skoraj podvoji.

**Izhodišče.** Najpreprostejša napoved: za vsako naročilo rečemo isto število, **mediano učnih podatkov**.
Napako izračunamo ročno in s funkcijama iz scikit-learn.

In [ ]:
mediana = y_ucni.median()
napaka = y_val - mediana                                  # za koliko dni smo se zmotili pri vsakem naročilu

print(f"Napoved za vsa naročila: {mediana:.1f} dni")
print(f"MAE  ročno: {napaka.abs().mean():.2f}    scikit-learn: {mean_absolute_error(y_val, np.full(len(y_val), mediana)):.2f}")
print(f"RMSE ročno: {np.sqrt((napaka ** 2).mean()):.2f}    scikit-learn: {root_mean_squared_error(y_val, np.full(len(y_val), mediana)):.2f}")

print(f"Najdaljša dostava: učna množica {y_ucni.max():.0f} dni, validacijska {y_val.max():.0f} dni;",
      "dostav, daljših od 60 dni: učna", int((y_ucni > 60).sum()), " validacijska", int((y_val > 60).sum()))

Z izhodiščem se zmotimo v povprečju za 6,5 dneva (MAE); RMSE je 9,2 dneva. Vsak model mora biti boljši od tega.

> **Povezava s 1. modulom:** mediana je odporna na ekstreme, zato smo jo izbrali za izhodišče. Zelo dolge dostave
> so osamelci, ki smo jih že srečali. RMSE je nanje občutljiva, MAE precej manj.

---
## Linearna regresija
### A3. Premica
Model je **premica**: $\hat{y} = b_0 + \beta_1 x$. Pri nas: `dni_dostave = presek + naklon × obljubljeni_dnevi`.
$b_0$ je presek, $\beta_1$ je naklon. Uporabimo jo kot vsak model: `fit`, nato `predict`.

In [ ]:
premica = LinearRegression().fit(X_ucni, y_ucni)

print(f"Korelacija obljubljeni_dnevi in dni_dostave: r = {ucni['obljubljeni_dnevi'].corr(ucni['dni_dostave']):.2f}")
print(f"presek b0 = {premica.intercept_:.2f},  naklon β1 = {premica.coef_[0]:.3f}")
print(f"Primer: obljubljenih 20 dni  →  napoved {premica.predict(pd.DataFrame({'obljubljeni_dnevi': [20]}))[0]:.1f} dni")
print(f"Delež učnih naročil, dostavljenih pred obljubljenim rokom: {(ucni['dni_dostave'] < ucni['obljubljeni_dnevi']).mean():.0%}")

Naklon modela pove: obljubljeni rok, ki je za en dan daljši, je povezan s približno 0,44 dneva daljšo napovedano dostavo.
To je povezava v podatkih, ne dokaz, da podaljšanje obljube povzroči počasnejšo dostavo.
Večina naročil v tej zbirki je dostavljena pred obljubljenim rokom.

> **Povezava s 1. modulom:** razsevni diagram z vzorcem in prosojnostjo ter korelacija. Korelacija pove, *kako močna*
> je povezava; naklon premice oceni, za koliko dni se spremeni napoved ob enem obljubljenem dnevu več.

Na grafu: siva pika je eno naročilo (vzorec 3 000 naročil, s prosojnostjo), modra črta je model.

In [ ]:
vzorec = ucni[["obljubljeni_dnevi", "dni_dostave"]].sample(3000, random_state=SEME)
crta = pd.DataFrame({"obljubljeni_dnevi": [7, 60]})
crta["napoved"] = premica.predict(crta)

(ggplot(vzorec, aes(x="obljubljeni_dnevi", y="dni_dostave"))
 + geom_point(alpha=0.15, size=1, color=SIVA)
 + geom_line(aes(y="napoved"), data=crta, color=MODRA, size=1.5)
 + coord_cartesian(xlim=(5, 60), ylim=(0, 60))
 + labs(title="Dni dostave glede na obljubljeni rok", x="obljubljeni dnevi", y="dni dostave"))

In [ ]:
def napake(y, napoved):
    return {"MAE": mean_absolute_error(y, napoved), "RMSE": root_mean_squared_error(y, napoved)}

pd.DataFrame({"izhodišče (mediana)": napake(y_val, np.full(len(y_val), mediana)),
              "premica": napake(y_val, premica.predict(X_val))}).T.round(2)

Premica je boljša od izhodišča: na validacijskih naročilih se zmoti v povprečju za 0,8 dneva manj.

### A4. Učenje = zmanjševanje napake
Kako je model našel prav ta naklon? Poskusimo veliko naklonov (presek pustimo pri miru) in za vsakega izračunamo
napako na **učnih** podatkih. Linearna regresija izbere premico z najmanjšo vsoto kvadratov napak — torej z najmanjšo RMSE.

In [ ]:
nakloni = np.linspace(0, 1, 41)
rmse = [root_mean_squared_error(y_ucni, premica.intercept_ + naklon * X_ucni["obljubljeni_dnevi"]) for naklon in nakloni]
poskusi = pd.DataFrame({"naklon": nakloni, "RMSE": rmse})

(ggplot(poskusi, aes(x="naklon", y="RMSE"))
 + geom_line(color=MODRA, size=1.2)
 + geom_vline(xintercept=premica.coef_[0], color=ORANZNA, linetype="dashed", size=1.2)
 + labs(title="Napaka na učnih podatkih glede na naklon premice",
        subtitle=f"oranžna črta: naklon, ki ga je izbral model ({premica.coef_[0]:.3f})", x="naklon", y="RMSE (dni)"))

Dno "doline" je pri naklonu modela. Tudi pri večjih modelih nastavitve izbiramo tako, da zmanjšujemo izbrano
funkcijo napake, čeprav se načini učenja med modeli razlikujejo.

### A5. Premica z več značilkami
Značilk je lahko več. Vsaka dobi svoj koeficient: $\hat{y} = b_0 + \beta_1 x_1 + \beta_2 x_2 + \beta_3 x_3$.
Obljubljenemu roku dodamo poštnino in delež postavk, ki pridejo iz iste zvezne države kot kupec.

In [ ]:
ZNACILKE = ["obljubljeni_dnevi", "postnina", "delez_postavk_ista_drzava"]
vec_znacilk = LinearRegression().fit(ucni[ZNACILKE], y_ucni)

print(f"presek b0 = {vec_znacilk.intercept_:.2f}")
print(f"Validacijska MAE: ena značilka {mean_absolute_error(y_val, premica.predict(X_val)):.2f}, "
      f"tri značilke {mean_absolute_error(y_val, vec_znacilk.predict(validacijski[ZNACILKE])):.2f}")
pd.DataFrame({"značilka": ZNACILKE, "koeficient β": vec_znacilk.coef_}).round(3)

Koeficient beremo tako: če se značilka poveča za 1 in se drugi dve ne spremenita, se napoved spremeni za β.
En obljubljeni dan več pomeni 0,31 dneva daljšo dostavo. Naročilo, pri katerem vse postavke pridejo iz iste zvezne države
kot kupec, je dostavljeno približno 3,7 dneva prej. Napaka se zmanjša s 5,64 na 5,43 dneva.

Stolpci so v shemi razvrščeni v **skupine**. Besedilni stolpci (zvezna država, kategorija) so v njih že zapisani kot stolpci 0/1.
V zvezku B boste sami izbirali, katere skupine dodate.

In [ ]:
skupine = shema["skupine_za_model"]                       # skupine stolpcev, pripravljenih za model
pd.DataFrame({"skupina": list(skupine), "stolpcev": [len(s) for s in skupine.values()],
              "prvi trije stolpci": [", ".join(s[:3]) for s in skupine.values()]})

---
## Polinomska regresija
### A6. Krivulja: dodamo stolpce x², x³
Povezava ni čisto ravna. Premici dovolimo, da se ukrivi: kot nove stolpce dodamo **x², x³ …**
Model stopnje 3 je $\hat{y} = b_0 + \beta_1 x + \beta_2 x^2 + \beta_3 x^3$. Stolpce doda funkcija `potence`.

In [ ]:
def potence(X, stopnja):
    """Vsakemu stolpcu doda njegove potence: x, x², x³ … do izbrane stopnje."""
    return pd.DataFrame({f"{c}^{k}": X[c] ** k for c in X.columns for k in range(1, stopnja + 1)})

trije = pd.DataFrame({"obljubljeni_dnevi": [10, 20, 30]})
potence(trije, 3)

Model je še vedno linearna regresija, le da ima več stolpcev. Naredimo tri korake. Model vedno učimo samo na **učnih** podatkih.

> **Povezava s 1. modulom:** potence so zelo velika števila (30³ = 27 000), zato jih pred učenjem **standardiziramo**.
> Povprečje in standardni odklon izračunamo samo iz učnih podatkov.

In [ ]:
P_ucni = potence(X_ucni, 3)                                              # 1. dodamo stolpca x² in x³
skala = StandardScaler().fit(P_ucni)                                     # 2. povprečje in odklon: samo iz učnih podatkov
krivulja = LinearRegression().fit(skala.transform(P_ucni), y_ucni)       # 3. linearna regresija na vseh treh stolpcih

napoved = krivulja.predict(skala.transform(potence(X_val, 3)))           # iste korake uporabimo na validacijskih podatkih
print(f"Stopnja 3: validacijska MAE {mean_absolute_error(y_val, napoved):.3f}   (premica {mean_absolute_error(y_val, premica.predict(X_val)):.3f})")

Iste tri korake zapišemo v funkcijo `nauci_polinom` (kopija je v `utils/orodja.py`). Vrne funkcijo `napovej`:
damo ji tabelo, vrne napovedi.

In [ ]:
def nauci_polinom(X, y, stopnja):
    """Nauči polinom izbrane stopnje. Vrne funkcijo napovej: damo ji tabelo, vrne napovedi."""
    skala = StandardScaler().fit(potence(X, stopnja))                          # povprečje in odklon: samo iz podanih (učnih) podatkov
    model = LinearRegression().fit(skala.transform(potence(X, stopnja)), y)    # linearna regresija na vseh potencah

    def napovej(X_novi):
        return model.predict(skala.transform(potence(X_novi, stopnja)))        # isti koraki na novih podatkih
    return napovej

krivulja_2 = nauci_polinom(X_ucni, y_ucni, 2)
krivulja_3 = nauci_polinom(X_ucni, y_ucni, 3)

pd.DataFrame({"premica (stopnja 1)": napake(y_val, premica.predict(X_val)),
              "stopnja 2": napake(y_val, krivulja_2(X_val)),
              "stopnja 3": napake(y_val, krivulja_3(X_val))}).T.round(3)

Vse modele narišemo na isti graf. Funkcija `graf_napovedi` (kopija je v `utils/orodja.py`) nariše črto za vsak model.
Sive pike so povprečja učnih podatkov: povprečen čas dostave v 40 pasovih obljubljenega roka.

In [ ]:
def graf_napovedi(modeli, podatki, x="obljubljeni_dnevi", y="dni_dostave", povprecja=True):
    """Napovedi več modelov na istem grafu. Vsak model uporablja eno značilko x.

    modeli = {ime: model}. Model je naučen model s funkcijo predict (premica, drevo)
    ali funkcija, ki za tabelo s stolpcem x vrne napovedi (tako funkcijo vrne nauci_polinom).
    Pike: povprečje y v 40 pasovih x (povprecja=True) ali vsaka vrstica posebej (False).
    """
    spodaj, zgoraj = podatki[x].quantile([0.01, 0.99]) if povprecja else (podatki[x].min(), podatki[x].max())
    znotraj = podatki[(podatki[x] >= spodaj) & (podatki[x] <= zgoraj)]
    pike = znotraj.groupby(pd.cut(znotraj[x], 40), observed=True)[[x, y]].mean() if povprecja else znotraj[[x, y]]
    mreza = pd.DataFrame({x: np.linspace(spodaj, zgoraj, 300)})
    crte = []
    for ime, model in modeli.items():
        napovej = model.predict if hasattr(model, "predict") else model      # naučen model ali funkcija
        crte.append(mreza.assign(napoved=napovej(mreza), model=ime))
    crte = pd.concat(crte)
    crte["model"] = pd.Categorical(crte["model"], list(modeli))
    barve = [MODRA, ORANZNA, ZELENA, "#8e44ad", "#555555"][:len(modeli)]
    return (ggplot()
            + geom_point(aes(x=x, y=y), data=pike, color=SIVA, size=2, alpha=0.8)
            + geom_line(aes(x=x, y="napoved", color="model"), data=crte, size=1.3)
            + scale_color_manual(values=barve)
            + labs(color="", subtitle="sive pike: " + ("povprečje v 40 pasovih x" if povprecja else "posamezne vrstice")))

(graf_napovedi({"premica (stopnja 1)": premica, "stopnja 2": krivulja_2, "stopnja 3": krivulja_3}, ucni)
 + labs(title="Premica in krivulji", x="obljubljeni dnevi", y="dni dostave"))

Krivulja se pikam prilega bolje kot premica: pri kratkih rokih dostava raste hitreje, pri dolgih se umiri.
Izboljšanje na validacijski množici je majhno: MAE 5,64 → 5,51.

### A7. Je višja stopnja še boljša?
Izmerimo napako za stopnje od 1 do 15, na učnih in na validacijskih podatkih.

In [ ]:
vrstice = []
for stopnja in range(1, 16):
    napovej = nauci_polinom(X_ucni, y_ucni, stopnja)
    vrstice.append({"stopnja": stopnja, "učna MAE": mean_absolute_error(y_ucni, napovej(X_ucni)),
                    "validacijska MAE": mean_absolute_error(y_val, napovej(X_val)),
                    "validacijska RMSE": root_mean_squared_error(y_val, napovej(X_val))})
po_stopnjah = pd.DataFrame(vrstice)
po_stopnjah.round(3)

In [ ]:
dolga = po_stopnjah.melt(id_vars="stopnja", value_vars=["učna MAE", "validacijska MAE"], var_name="podatki", value_name="MAE")

(ggplot(dolga, aes(x="stopnja", y="MAE", color="podatki"))
 + geom_line(size=1.2)
 + geom_point(size=2.5)
 + scale_color_manual(values=[MODRA, ORANZNA])
 + labs(title="Napaka glede na stopnjo polinoma", x="stopnja polinoma", y="MAE (dni)", color=""))

Stopnja 2 je na validacijski množici boljša od premice. Od stopnje 2 do 6 so razlike majhne (MAE 5,50–5,52);
pri višjih stopnjah je napaka nekoliko večja (5,53–5,54). Povečevanje stopnje torej tu ne prinese pomembnega izboljšanja. Učna napaka se medtem še malo zmanjša. Višja stopnja torej ne pomeni boljšega modela.

### A8. Skrajni primer: malo podatkov, visoka stopnja
Pri več kot 50 000 naročilih se krivulja težko "nauči na pamet". Kaj pa, če imamo samo **40 naročil**?
To je namenoma skrajen primer, da učinek dobro vidimo.

In [ ]:
malo = ucni.sample(40, random_state=8)                      # samo 40 učnih naročil
X_malo, y_malo = malo[["obljubljeni_dnevi"]], malo["dni_dostave"]

mirna = nauci_polinom(X_malo, y_malo, 2)
divja = nauci_polinom(X_malo, y_malo, 10)

# validacijska naročila z obljubljenim rokom znotraj razpona teh 40 naročil
znotraj = validacijski[validacijski["obljubljeni_dnevi"].between(X_malo["obljubljeni_dnevi"].min(), X_malo["obljubljeni_dnevi"].max())]
print(f"Razpon obljubljenih dni v vzorcu: {X_malo['obljubljeni_dnevi'].min():.1f}–{X_malo['obljubljeni_dnevi'].max():.1f}; "
      f"validacijskih naročil v tem razponu: {len(znotraj)} od {len(validacijski)}")
print("Štirje najdaljši obljubljeni roki v vzorcu:", malo["obljubljeni_dnevi"].nlargest(4).round(1).tolist())

pd.DataFrame({"učna MAE (40 naročil)": [mean_absolute_error(y_malo, napovej(X_malo)) for napovej in (mirna, divja)],
              "validacijska MAE": [mean_absolute_error(znotraj["dni_dostave"], napovej(znotraj[["obljubljeni_dnevi"]])) for napovej in (mirna, divja)]},
             index=["stopnja 2", "stopnja 10"]).round(2)

In [ ]:
(graf_napovedi({"stopnja 2": mirna, "stopnja 10": divja}, malo, povprecja=False)
 + coord_cartesian(ylim=(0, 45))
 + labs(title="40 naročil: mirna in divja krivulja", x="obljubljeni dnevi", y="dni dostave"))

Krivulja stopnje 10 gre bliže učnim pikam (manjša učna napaka), na novih naročilih pa je približno dvakrat slabša.
Zvija se med posameznimi pikami, in kjer v vzorcu ni naročil (med 39 in 51 obljubljenimi dnevi), podivja.
To je **pretirano prilagajanje** — isto, kot smo videli pri globokem drevesu.

---
Naprej: zvezek **`B_regresija`**. V nadaljevanju bloka na isti graf dodamo še regresijsko drevo.